# Hands-on Tutorial: Automated Machine Learning with auto-sklearn

This notebook provides a practical guide to using `auto-sklearn`, a powerful library for automated machine learning. We will cover the basics of using `auto-sklearn` for both classification and regression tasks, as well as how to control the optimization process.

## 1. Installation

`auto-sklearn` (the original library) is a powerful AutoML system (`CASH + Bayesian optimization + ensembling + meta-learning warm-start`)
Unfortunately, auto-sklearn has technical limitations.

* Linux/macOS is recommended. On Windows, installation is often painful or unsupported because of compiled dependencies and build toolchains.

* It tends to work best with older Python stacks (commonly Python 3.9 or 3.10, depending on the build). Newer Python versions frequently break wheels/compilation.

* It depends on native libraries (C/C++ builds), so “pip install” is not always enough unless you’re on a supported OS/Python combo.

In this session, we will use a simpler version that does not faithfully do the meta-learning and the rest of the staff.

If you want to try the power of `auto-sklearn`, you can try on a Docker container.

`dockerfile`
```
FROM python:3.9-slim

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential swig \
 && rm -rf /var/lib/apt/lists/*

RUN pip install --no-cache-dir -U pip setuptools wheel
RUN pip install --no-cache-dir auto-sklearn

WORKDIR /workspace
```

`Buid + run`

```
docker build -t autosklearn39 .
docker run -it --rm -v %cd%:/workspace autosklearn39 bash   # Windows PowerShell
# docker run -it --rm -v $(pwd):/workspace autosklearn39 bash  # macOS/Linux

```

`then in the container`

`python -c "import autosklearn; print(autosklearn.__version__)"`

In this session, we will use auto-sklearn2

In [ ]:
!pip -q install -U auto-sklearn2

In [ ]:
import auto_sklearn2
#import autosklearn.classification
print(auto_sklearn2 .__version__)

1.0.0


In [ ]:
from auto_sklearn2 import AutoSklearnClassifier
import inspect
print(inspect.signature(AutoSklearnClassifier))

(time_limit=120, n_jobs=-1, random_state=None)


## 2. Classification Example

Let's start with a simple classification task. We will use the breast cancer dataset from scikit-learn.

In [ ]:
import sklearn.datasets
import sklearn.model_selection
import sklearn.metrics
from auto_sklearn2 import AutoSklearnClassifier


# Load the dataset
X, y = sklearn.datasets.load_breast_cancer(return_X_y=True,as_frame=True)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, random_state=1)

# Instantiate and fit the classifier
automl = AutoSklearnClassifier(
    time_limit=120,  # seconds
    #per_run_time_limit=30,  # seconds
    #tmp_folder="/tmp/autosklearn_classification_example_tmp",
    n_jobs=-1,

)
automl.fit(X_train, y_train)

# Evaluate the model
y_hat = automl.predict(X_test)
print("Accuracy score:", sklearn.metrics.accuracy_score(y_test, y_hat))

# Print the best model
print("Best model:", automl.best_params)


ERROR:auto_sklearn2:Error evaluating standard_scaler + multinomial_nb: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrap

Accuracy score: 0.965034965034965
Best model: {'preprocessor': 'robust_scaler', 'classifier': 'logistic_regression'}


## 3. Regression Example

Now, let's try a regression task. We will use the Boston housing dataset.

In [ ]:
from sklearn.datasets import fetch_california_housing
import sklearn.model_selection
import sklearn.metrics
from auto_sklearn2 import AutoSklearnRegressor

# Load the dataset
X, y = fetch_california_housing(return_X_y=True, as_frame=False)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, random_state=1)

# Instantiate and fit the regressor
automl = AutoSklearnRegressor(
    time_limit=120,  # seconds
    # per_run_time_limit=30,  # seconds
    # tmp_folder="/tmp/autosklearn_regression_example_tmp",
    n_jobs=-1,

)
automl.fit(X_train, y_train)

# Evaluate the model
y_hat = automl.predict(X_test)
print("R2 score:", sklearn.metrics.r2_score(y_test, y_hat))

# Print the best model
print("Best model:", automl.best_params)


R2 score: 0.7993277952284603
Best model: {'preprocessor': 'standard_scaler', 'regressor': 'random_forest'}
